In [ ]:
import torch
from torch import nn
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

#Import matplotlib for visualizatiion
import matplotlib.pyplot as plt
#check versions
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

## 1.Getting a Dataset

The dataset we'll be using is FashionMNIST from torchvision datasets



In [ ]:
from torchvision import datasets
train_data=datasets.FashionMNIST(
    root="data", # where to download data to
    train=True,# do we want training datasets?
    download=True,
    transform=torchvision.transforms.ToTensor(),
    target_transform=None
)

test_data=datasets.FashionMNIST(
    root="data", # where to download data to
    train=False,# do we want training datasets?
    download=True,
    transform=torchvision.transforms.ToTensor(),
    target_transform=None
)

In [ ]:
len(train_data),len(test_data)

In [ ]:
# See the first training example

image, label=train_data[0]
image,label

In [ ]:
class_names=train_data.classes
class_names

In [ ]:
class_to_idx=train_data.class_to_idx
class_to_idx

In [ ]:
image.shape,label

### 1.2 Visuaizing the data




In [ ]:
import matplotlib.pyplot as plt

image,label=train_data[0]
plt.imshow(image.squeeze())
plt.title(label);

In [ ]:
plt.imshow(image.squeeze(),cmap="gray")
plt.title(class_names[label]);
plt.axis(False);

In [ ]:
#plot more images
torch.manual_seed(42)
fig=plt.figure(figsize=(9,9))
rows,cols=4,4
for i in range(1,rows*cols+1):
  # print(i)
  random_idx=torch.randint(0,len(train_data),size=[1]).item()
  # print(random_idx)
  image,label=train_data[random_idx]
  fig.add_subplot(rows,cols,i)
  plt.imshow(image.squeeze(),cmap="gray")
  plt.title(class_names[label])
  plt.axis(False)

## 2.Prepare DataLoader
Right now,our data is in form of Pytorch Datasets.
DataLoader turns our datasets into iterables.
More specifically we want to turn our datasets into batches.
why would we do this?
1. It is more computtationaly efficient, as in, your computing hardware may not be able to look at 60000 images in onbe hit.
So we brreak it down to 32 imnages at a time
2.It gives our neural network more chance to upgrade its gradients per epoch.

In [ ]:
from torch.utils.data import DataLoader

#Setup the batch size hyperParameter

BATCH_SIZE=32

#Turn datatsets into iterables
train_dataloader=DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

#Turn datatsets into iterables
test_dataloader=DataLoader(
    dataset=test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_dataloader,train_dataloader

In [ ]:
len(train_dataloader),len(test_dataloader)

In [ ]:
train_features_batch,train_labels_batch=next(iter(train_dataloader))
train_features_batch.shape,train_labels_batch.shape

In [ ]:
#Show a sample
torch.manual_seed(42)
random_idx=torch.randint(0,len(train_features_batch),size=[1]).item()
img,label=train_features_batch[random_idx],train_labels_batch[random_idx]

plt.imshow(img.squeeze() ,cmap='gray')
plt.title(class_names[label])
plt.axis(False)



## 3.Model 0: Build a baseline Model
When  Starting to build a series of machine learning modelling experiments ,its best practice to start with a baseline model.'

A baseline model is a simple model you will try and improve upon with subsequent models/experiments.

In other words: Start simply and add complexityy when necessary

In [ ]:
#Creating a flatten Layer
flatten_model=nn.Flatten()

# Get a single sample
x=train_features_batch[0]
# x.shape

#Flatten the sample
output=flatten_model(x)
output.shape


In [ ]:
from torch import nn
class FashionMNISTModelV0(nn.Module):
  def __init__(self,
               input_shape: int,
               hidden_units: int,
               output_shape:int):
    super().__init__()
    self.layer_stack=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=input_shape,out_features=hidden_units),
        nn.Linear(in_features=hidden_units,out_features=output_shape)
    )

  def forward(self,x):
    return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)


#Setup model with input parameteres
model_0=FashionMNISTModelV0(
    input_shape=28*28,
    hidden_units=10,
    output_shape=len(class_names)
).to("cpu")

model_0

In [ ]:
dummy_x=torch.rand([1,1,28,28])

model_0(dummy_x).shape

### 3.1 Setup a loss function and optimizer and evaluation matrix

* Loss Function-since we're working with multi-class data,our loss function will
bw `nn.CrossEntropyLoss()`
* optimizer-our optimizer `torch.optim.SGD()`
* Evaluation Matrix- We'll evaluate our model by accuracy

In [ ]:
import requests
from pathlib import Path

#Download helper functions from learn Pytorch repo
if Path("helper_functions.py").is_file():
  print("helper_functions.py exists")
else:
  print("Downloading helper_functions.py")
  request=requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py","wb") as f:
    f.write(request.content)


In [ ]:
# Import accuracy metric
from helper_functions import accuracy_fn

#Setup loss function and optimizer
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(params=model_0.parameters(),
                          lr=0.1)

### 3.2 Creating a function to time our experiments
Machine Learning is very experimental

Two of the Main Things you'll often want to track

* Model's Performance (loss and accuracy values etc)
* How fast it runs


In [ ]:
from timeit import default_timer as timer
def print_train_time(start:float,
                     end:float,
                     device: torch.device=None):
  total_time=end-start
  print(f"Time to train model on device {device}: {total_time:.3f} seconds")
  return total_time

In [ ]:
start_time=timer()
#some code....

end_time=timer()
print_train_time(start=start_time,end=end_time,device="cpu")

### 3.3 Creating a trianing loop and training a model on batches of data

1. Loop through epochs.
2. Loop through training batches,perform training steps ,calculate the train loss per batch.
3. Loop through testing batches,perform testing steps , calculate the test loss per batch.
4. Print out what's happening
5. time it all( for fun).

In [ ]:
from tqdm.auto import tqdm

#Set the seed and satrt the timer
torch.manual_seed(42)
train_time_start_on_cpu=timer()

# Set the number of epochs
epochs=3

#Create a trianing loop
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n-------")
  train_loss=0
  # Add a loop to loop through the training batches
  for batch,(X,y) in enumerate(train_dataloader):
    model_0.train()
    #1.Forward
    y_pred=model_0(X)
    #2.Calculate the loss
    loss=loss_fn(y_pred,y)
    train_loss+=loss

    optimizer.zero_grad()
    #3.Optimizer zero grad
    loss.backward()
    #4.Optimizer step
    optimizer.step()

    #Print out what's happening
    if batch % 400 ==0:
      print(f"Looked at {batch*len(X)}/{len(train_dataloader.dataset)} samples")

  train_loss/=len(train_dataloader)
  print(f"Train loss: {train_loss:.5f}")


  ### Testing
  test_loss,test_acc=0,0
  model_0.eval()
  with torch.inference_mode():
    for X_test,y_test in test_dataloader:
      test_pred=model_0(X_test)

      #2.Calculate loss
      test_loss+=loss_fn(test_pred,y_test)

      #3.Calculate accuracy
      test_acc+=accuracy_fn(y_true=y_test,y_pred=test_pred.argmax(dim=1))

    test_loss/=len(test_dataloader)
    test_acc/=len(test_dataloader)

  print(f"\nTrain loss:{train_loss:.4f} | Test loss:{test_loss:.4f} | Test acc:{test_acc:.4f}")

train_time_end_on_cpu=timer()

total_train_time_model_0=print_train_time(start=train_time_start_on_cpu,end=train_time_end_on_cpu,device=str(next(model_0.parameters()).device))

In [ ]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)

def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               accuracy_fn,
               device:torch.device):
    model.eval()
    model.to(device)  # ensure model on device [5]
    total_loss, total_acc = 0.0, 0.0
    count_batches = 0
    with torch.inference_mode():
        for X, y in data_loader:
            X, y = X.to(device), y.to(device)  # move batch [3][4]
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            total_loss += loss.item()  # scalar accumulation [4][6]
            preds = y_pred.argmax(dim=1)
            total_acc += accuracy_fn(y_true=y, y_pred=preds)
            count_batches += 1
    avg_loss = total_loss / max(count_batches, 1)
    avg_acc = total_acc / max(count_batches, 1)
    return {
        "model_name": model.__class__.__name__,
        "model_loss": avg_loss,
        "model_acc": avg_acc
    }


model_0_results=eval_model(model=model_0,
                            data_loader=test_dataloader,
                            loss_fn=loss_fn,
                            accuracy_fn=accuracy_fn,
                           device=device)

model_0_results


In [ ]:
!nvidia-smi

In [ ]:
torch.cuda.is_available()

In [ ]:
#Setup Device Agnostic code
import torch
device="cuda" if torch.cuda.is_available() else "cpu"
device

### 6. Model1:Building a Better model with non-linearity

In [ ]:
#Create a model with non-linear and linear layers

class FashionMNISTModelV1(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()  # proper base init
        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=input_shape, out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units, out_features=output_shape),
            nn.ReLU() # no ReLU here for logits
        )

    def forward(self, x: torch.Tensor):
        return self.layer_stack(x)


In [ ]:
next(model_0.parameters()).device

In [ ]:
torch.manual_seed(42)

model_1=FashionMNISTModelV1(input_shape=784,
                            hidden_units=10,
                            output_shape=len(class_names)).to(device)
next(model_1.parameters()).device

### 6.1 Setup Loss Function and optimizer


In [ ]:
!wget https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py

In [ ]:
from helper_functions import accuracy_fn
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(params=model_1.parameters(),
                          lr=0.1)

### 6.2 Functionizing training and evaluation/testing loop

In [ ]:
def train_step(model:torch.nn.Module,
               data_loader:torch.utils.data.DataLoader,
               loss_fn:torch.nn.Module,
               optimizer:torch.optim.Optimizer,
               accuracy_fn,
               device:torch.device=device):

  model.train()
  train_loss ,train_acc=0,0
  # Add a loop to loop through the training batches
  for batch,(X,y) in enumerate(data_loader):
    #Put Data in target device
    X,y=X.to(device),y.to(device)
    #1.Forward
    y_pred=model(X)
    #2.Calculate the loss
    loss=loss_fn(y_pred,y)
    train_loss+=loss
    train_acc+=accuracy_fn(y_true=y,y_pred=y_pred.argmax(dim=1)) #go from logits --> prediction label
    optimizer.zero_grad()
    #3.Optimizer zero grad
    loss.backward()
    #4.Optimizer step
    optimizer.step()


  train_loss/=len(data_loader)
  train_acc/=len(data_loader)
  print(f"Train loss: {train_loss:.5f} | Train acc: {train_acc:.2f}%\n")

In [ ]:
def test_step(model:torch.nn.Module,
              data_loader:torch.utils.data.DataLoader,
              loss_fn:torch.nn.Module,
              accuracy_fn,
              device:torch.device=device):
  test_loss,test_acc=0,0

  #Put the model in eval
  model.eval()

  #Turn on the inference mode context manager
  with torch.inference_mode():
    for X,y in data_loader:
      X,y=X.to(device),y.to(device)

      #1.Forward Pass
      test_pred=model(X)

      #2. Calculate the loss/acc
      test_loss+=loss_fn(test_pred,y)
      test_acc+=accuracy_fn(y_true=y,
                         y_pred=test_pred.argmax(dim=1)) #go from logits-->prediction labels

    test_loss/=len(data_loader)
    test_acc/=len(data_loader)

    print(f"Test loss: {test_loss:.5f} | Test acc: {test_acc:.2f}%\n")

In [ ]:
torch.manual_seed(42)

#Measure Time
from timeit import default_timer as timer
train_time_start_on_gpu=timer()

#Set the epochs
epochs=3


from helper_functions import accuracy_fn

#Setup loss function and optimizer
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(params=model_1.parameters(),
                          lr=0.1)
#Create a optimization and evaluation loop using train_step() and test_step()

for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n-------")
  train_step(model=model_1,
             data_loader=train_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             accuracy_fn=accuracy_fn,
             device=device)
  test_step(model=model_1,
            data_loader=test_dataloader,
            loss_fn=loss_fn,
            accuracy_fn=accuracy_fn,
            device=device)
train_time_end_on_gpu=timer()
total_train_time_model_1=print_train_time(start=train_time_start_on_gpu,end=train_time_end_on_gpu,device=str(next(model_1.parameters()).device))

## Model 2: Building a convolutional neural Network(CNN)

CNN's are also knownn as Convnets.
</br>
CNN's are known for their capabilities to find patterns in visual
</br>
To find out whats happening inside a CNN, see the website-https://poloclub.github.io/cnn-explainer/

In [ ]:
#create a convoulutional neural network

class FashionMNISTModelV2(nn.Module):
  """
  Model architecture that replicates the TinyVGG
  model from CNN explainer website.
  """
  def __init__(self,input_shape:int,hidden_units:int,output_shape:int):
    super().__init__()
    self.conv_block_1=nn.Sequential(
        nn.Conv2d(in_channels=input_shape,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )


    self.conv_block_2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,
                  out_channels=hidden_units,
                  kernel_size=3,
                  stride=1,
                  padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2)
    )

    self.classifier=nn.Sequential(
        nn.Flatten(),
        nn.Linear(in_features=hidden_units*7*7,
                  out_features=output_shape)
    )
  def forward(self,x):
    x=self.conv_block_1(x)
    x=self.conv_block_2(x)
    x=self.classifier(x)
    return x


In [ ]:
image.shape

In [ ]:
torch.manual_seed(42)
model_2=FashionMNISTModelV2(input_shape=1,
                            hidden_units=10,
                            output_shape=len(class_names)).to(device)

###7.1 Stepping Through `nn.Conv2d()`

In [ ]:
torch.manual_seed(42)


#create a  batch of images

images=torch.randn(size=(32,3,64,64))
test_image=images[0]

print(f"Image batch shape:{images.shape}")
print(f"Single image shape:{test_image.shape}")
print(f"Single image pixel values:\n{test_image}")


In [ ]:
#create a single conv2d layer
torch.manual_seed(42)
conv_layer=nn.Conv2d(in_channels=3,
                     out_channels=10,
                     kernel_size=3,
                     stride=1,
                     padding=0)



#Pass the data to conv layer

conv_output=conv_layer(test_image.unsqueeze(dim=0))
conv_output.shape

###7.2 Stepping through `nn.MaxPool2d()`

In [ ]:
# Print out original image shape without and with unsqueezed dimension
print(f"Test image original shape: {test_image.shape}")
print(f"Test image with unsqueezed dimension: {test_image.unsqueeze(dim=0).shape}")

# Create a sample nn.MaxPoo2d() layer
max_pool_layer = nn.MaxPool2d(kernel_size=2)

# Pass data through just the conv_layer
test_image_through_conv = conv_layer(test_image.unsqueeze(dim=0))
print(f"Shape after going through conv_layer(): {test_image_through_conv.shape}")

# Pass data through the max pool layer
test_image_through_conv_and_max_pool = max_pool_layer(test_image_through_conv)
print(f"Shape after going through conv_layer() and max_pool_layer(): {test_image_through_conv_and_max_pool.shape}")

### 7.3 Setup a loss function and optimizer for `model_2`

In [ ]:
from helper_functions import accuracy_fn

loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.SGD(params=model_2.parameters(),
                          lr=0.1)

### 7.4 Training and testing `model_2` using our training and test functions

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)


#Measure the time
from timeit import default_timer as timer
train_time_start=timer()

#Train and Test model

epochs=3
for epoch in tqdm(range(epochs)):
  print(f"Epoch: {epoch}\n-------")
  train_step(model=model_2,
             data_loader=train_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             accuracy_fn=accuracy_fn,
             device=device)
  test_step(model=model_2,
            data_loader=test_dataloader,
            loss_fn=loss_fn,
            accuracy_fn=accuracy_fn,
            device=device)

train_time_end=timer()

total_train_time=print_train_time(start=train_time_start,end=train_time_end,device=device)


In [ ]:
#Get the results
model_2_results=eval_model(
    model=model_2,
    data_loader=test_dataloader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device
)

model_2_results

## 8. Make and evaluate therandom predictions with best model

In [ ]:
def make_predictions(model: torch.nn.Module, data: list, device: torch.device = device):
    pred_probs = []
    model.eval()
    with torch.inference_mode():
        for sample in data:
            # Prepare sample
            sample = torch.unsqueeze(sample, dim=0).to(device) # Add an extra dimension and send sample to device

            # Forward pass (model outputs raw logit)
            pred_logit = model(sample)

            # Get prediction probability (logit -> prediction probability)
            pred_prob = torch.softmax(pred_logit.squeeze(), dim=0) # note: perform softmax on the "logits" dimension, not "batch" dimension (in this case we have a batch size of 1, so can perform on dim=0)

            # Get pred_prob off GPU for further calculations
            pred_probs.append(pred_prob.cpu())

    # Stack the pred_probs to turn list into a tensor
    return torch.stack(pred_probs)

In [ ]:
img,label=test_data[0][:10]
img.shape,label

In [ ]:
import random
# random.seed(42)
test_samples = []
test_labels = []
for sample, label in random.sample(list(test_data), k=9):
    test_samples.append(sample)
    test_labels.append(label)

# View the first test sample shape and label
print(f"Test sample image shape: {test_samples[0].shape}\nTest sample label: {test_labels[0]} ({class_names[test_labels[0]]})")

In [ ]:

plt.imshow(test_samples[0].squeeze(),cmap="gray")
plt.title(class_names[test_labels[0]])

In [ ]:
# Make predictions on test samples with model 2
pred_probs= make_predictions(model=model_2,
                             data=test_samples)

# View first two prediction probabilities list
pred_probs[:2]

In [ ]:
test_labels

In [ ]:
#Convert prediction probablities to labels
pred_labels=torch.argmax(pred_probs,dim=1)
pred_labels

In [ ]:
#plot predictions

plt.figure(figsize=(9,9))
nrows=3
ncols=3
for i,sample in enumerate(test_samples):
  plt.subplot(nrows,ncols,i+1)
  plt.imshow(sample.squeeze(),cmap="gray")

  #find the pred label
  pred_label=class_names[pred_labels[i]]

  #Get the truth label
  truth_label=class_names[test_labels[i]]

  text_title=f"Pred: {pred_label} | Truth: {truth_label}"

  if pred_label==truth_label:
    plt.title(text_title,fontsize=10,color="green")
  else:
    plt.title(text_title,fontsize=10,color="red")

  plt.axis(False)

### Making a Confusion MAtrisx for Evaluation

1. we need to make predictions with our trained model on custom dataset
2. Make a confusion matrix `torchmetrics.ConfusionMatrix`


In [ ]:
#Import tqdm for progress bar
from tqdm.auto import tqdm

# 1. Make predictions with trained model
y_preds = []
model_2.eval()
with torch.inference_mode():
  for X, y in tqdm(test_dataloader, desc="Making predictions"):
    # Send data and targets to target device
    X, y = X.to(device), y.to(device)
    # Do the forward pass
    y_logit = model_2(X)
    # Turn predictions from logits -> prediction probabilities -> predictions labels
    y_pred = torch.softmax(y_logit, dim=1).argmax(dim=1) # note: perform softmax on the "logits" dimension, not "batch" dimension (in this case we have a batch size of 32, so can perform on dim=1)
    # Put predictions on CPU for evaluation
    y_preds.append(y_pred.cpu())
# Concatenate list of predictions into a tensor
y_pred_tensor = torch.cat(y_preds)

In [ ]:
try:
    import torchmetrics, mlxtend
    print(f"mlxtend version: {mlxtend.__version__}")
    assert int(mlxtend.__version__.split(".")[1]) >= 19, "mlxtend verison should be 0.19.0 or higher"
except:
    !pip install -q torchmetrics -U mlxtend # <- Note: If you're using Google Colab, this may require restarting the runtime
    import torchmetrics, mlxtend
    print(f"mlxtend version: {mlxtend.__version__}")

In [ ]:
# Import mlxtend upgraded version
import mlxtend
print(mlxtend.__version__)
assert int(mlxtend.__version__.split(".")[1]) >= 19 # should be version 0.19.0 or higher

In [ ]:
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

# 2. Setup confusion matrix instance and compare predictions to targets
confmat = ConfusionMatrix(num_classes=len(class_names), task='multiclass')
confmat_tensor = confmat(preds=y_pred_tensor,
                         target=test_data.targets)

# 3. Plot the confusion matrix
fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(), # matplotlib likes working with NumPy
    class_names=class_names, # turn the row and column labels into class names
    figsize=(10, 7)
);

In [ ]:
from pathlib import Path

# Create models directory (if it doesn't already exist), see: https://docs.python.org/3/library/pathlib.html#pathlib.Path.mkdir
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, # create parent directories if needed
                 exist_ok=True # if models directory already exists, don't error
)

# Create model save path
MODEL_NAME = "03_pytorch_computer_vision_model_2.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_2.state_dict(), # only saving the state_dict() only saves the learned parameters
           f=MODEL_SAVE_PATH)

In [ ]:
# Create a new instance of FashionMNISTModelV2 (the same class as our saved state_dict())
# Note: loading model will error if the shapes here aren't the same as the saved version
loaded_model_2 = FashionMNISTModelV2(input_shape=1,
                                    hidden_units=10, # try changing this to 128 and seeing what happens
                                    output_shape=10)

# Load in the saved state_dict()
loaded_model_2.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

# Send model to GPU
loaded_model_2 = loaded_model_2.to(device)

In [ ]:
# Evaluate loaded model
torch.manual_seed(42)

loaded_model_2_results = eval_model(
    model=loaded_model_2,
    data_loader=test_dataloader,
    loss_fn=loss_fn,
    accuracy_fn=accuracy_fn,
    device=device
)

loaded_model_2_results

In [ ]:
# Check to see if results are close to each other (if they are very far away, there may be an error)
torch.isclose(torch.tensor(model_2_results["model_loss"]),
              torch.tensor(loaded_model_2_results["model_loss"]),
              atol=1e-08, # absolute tolerance
              rtol=0.0001)

In [ ]:
!pip install nbstripout
!nbstripout pytorch_computer_vision.ipynb
